# הערכה סופית וחד־פעמית על חלוקת Test

מחברת זו פותחת את חלוקת 2026 Q2 פעם אחת לאחר שכל החלטות המודל ננעלו. המודל משתמש רק ב־Elo קנוני, משום שהכוונון והתוספות שנבדקו לא עברו את רף הראיות שנקבע על Validation. המדדים המתקבלים כאן הם מדדי ה־holdout הסופיים של הפרויקט ואינם משמשים לשינוי פיצ'רים, פרמטרים או סף החלטה.

In [1]:
from pathlib import Path
import hashlib
import sys

import joblib
import numpy as np
import pandas as pd
from sklearn.metrics import accuracy_score, brier_score_loss, log_loss

project_root = Path.cwd()
if not (project_root / "data").exists():
    project_root = project_root.parent
elo_module_dir = project_root / "src" / "features"
if str(elo_module_dir) not in sys.path:
    sys.path.insert(0, str(elo_module_dir))
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from elo import PointInTimeEngine
from src.data.splitter import ChronologicalSplitter

RANDOM_SEED = 42

## יצירת פיצ'רים נקודתיים בזמן

המנוע עובר על כל הדאטה לפי `datetime`, ולאחר מכן לפי `match_id` ו־`game_id` כדי לשמור על סדר המפות בתוך סדרה. לכל שורה נשמרים דירוגי Global ו־Map שהיו זמינים לפני המפה. H2H ו־Rolling Form נשארים במימוש לשימוש עתידי, אך אינם נכנסים ל־allowlist של המודל.

מיקום המפה מחושב לאחר המיון: המפה הראשונה בכל `match_id` מסומנת כ־1 וכל מפה נוספת כ־2 ומעלה. הפילוח חשוב משום שמפות מאוחרות עשויות ליהנות מעדכון Elo בתוך הסדרה, בעוד מפה 1 מייצגת בצורה הישירה ביותר תחזית טרום־סדרתית.

In [2]:
source_columns = [
    "match_id", "game_id", "team1_id", "team1",
    "team2_id", "team2", "is_total", "bestOf",
    "score1_game", "score2_game", "map_name", "datetime",
    "team1_win", "team1_join_key", "team2_join_key",
]
source_df = pd.read_csv(
    project_root / "data" / "final_tournament_features.csv",
    usecols=source_columns, low_memory=False,
)
point_in_time_df = PointInTimeEngine(k_factor=24, initial_rating=1500).transform(source_df)
point_in_time_df["map_position"] = (
    point_in_time_df.groupby("match_id", sort=False).cumcount() + 1
)
splits = ChronologicalSplitter().split(point_in_time_df)
train_base = splits["train"]
val_base = splits["val"]
test_base = splits["test"]

print(f"Train גולמי: {len(train_base):,}")
print(f"Validation גולמי: {len(val_base):,}")
print(f"Test גולמי: {len(test_base):,}")
print(f"משחקים ייחודיים ב־Test: {test_base['match_id'].nunique():,}")
print(f"מפה 1 ב־Test: {test_base['map_position'].eq(1).sum():,}")
print(f"מפות 2+ ב־Test: {test_base['map_position'].gt(1).sum():,}")

Train גולמי: 5,472
Validation גולמי: 633
Test גולמי: 595
משחקים ייחודיים ב־Test: 294
מפה 1 ב־Test: 294
מפות 2+ ב־Test: 301


## סימטריזציה זהה בכל חלוקה

Train, Validation ו־Test מסומטרים כל אחד בנפרד. בעותק המשוקף הקבוצות ודירוגיהן מוחלפים, התווית מתהפכת והפרשי ה־Elo מחושבים מחדש. בדיקה אנטי־סימטרית דורשת שההפרשים יתהפכו בסימן בדיוק. שום שורה אינה עוברת בין חלוקות.

מטריצת המודל כוללת בדיוק שישה פיצ'רים: ארבעת דירוגי ה־Elo הגולמיים ושני ההפרשים. מיקום המפה ומזהי המשחק נשמרים לצורך פילוח ואינם משמשים לאימון.

In [3]:
rating_pairs = [
    ("team1_elo_global", "team2_elo_global"),
    ("team1_elo_map", "team2_elo_map"),
]
identity_pairs = [
    ("team1_id", "team2_id"),
    ("team1", "team2"),
    ("team1_join_key", "team2_join_key"),
]
absolute_columns = [column for pair in rating_pairs for column in pair]
feature_columns = absolute_columns + ["elo_global_diff", "elo_map_diff"]
context_columns = [
    "match_id", "game_id", "datetime", "map_name", "map_position",
    "team1_id", "team1", "team2_id", "team2",
    "team1_join_key", "team2_join_key", "team1_win",
]

def symmetrize_elo(split_df):
    original = split_df[context_columns + absolute_columns].copy().reset_index(drop=True)
    mirrored = original.copy()
    for team1_column, team2_column in identity_pairs + rating_pairs:
        mirrored[team1_column] = original[team2_column].to_numpy(copy=True)
        mirrored[team2_column] = original[team1_column].to_numpy(copy=True)
    mirrored["team1_win"] = 1 - original["team1_win"].to_numpy()

    symmetric = pd.concat([original, mirrored], ignore_index=True)
    symmetric["elo_global_diff"] = (
        symmetric["team1_elo_global"] - symmetric["team2_elo_global"]
    )
    symmetric["elo_map_diff"] = (
        symmetric["team1_elo_map"] - symmetric["team2_elo_map"]
    )
    midpoint = len(original)
    for diff_column in ["elo_global_diff", "elo_map_diff"]:
        original_values = symmetric.iloc[:midpoint][diff_column].to_numpy()
        mirrored_values = symmetric.iloc[midpoint:][diff_column].to_numpy()
        if not np.allclose(original_values, -mirrored_values):
            raise AssertionError(f"Diff sign did not flip: {diff_column}")
    if not np.isclose(symmetric["team1_win"].mean(), 0.5):
        raise AssertionError("Symmetrization did not balance the target.")
    return symmetric

train_df = symmetrize_elo(train_base)
val_df = symmetrize_elo(val_base)
test_df = symmetrize_elo(test_base)
train_val_df = pd.concat([train_df, val_df], ignore_index=True)

X_train_val = train_val_df[feature_columns]
y_train_val = train_val_df["team1_win"].astype(int)
X_test = test_df[feature_columns]
y_test = test_df["team1_win"].astype(int)

print(f"Train + Validation מסומטרים: {len(train_val_df):,}")
print(f"Test מסומטר: {len(test_df):,}")
print(f"פיצ'רים נעולים: {feature_columns}")

Train + Validation מסומטרים: 12,210
Test מסומטר: 1,190
פיצ'רים נעולים: ['team1_elo_global', 'team2_elo_global', 'team1_elo_map', 'team2_elo_map', 'elo_global_diff', 'elo_map_diff']


## תיקון מקור המודל: הערכת הארטיפקט שנפרס בפועל

הגרסה הקודמת של המחברת אימנה מודל גולמי מחדש על Train ו־Validation יחד, בעוד שמערכת הסימולציה משתמשת בארטיפקט אחר: מודל שאומן על Train בלבד וכיול איזוטוני שהותאם על Validation בלבד. לכן המדד הישן תיאר מודל שלא שירת תחזיות חיות. זהו תיקון provenance חד־פעמי: אנו טוענים את `canonical_elo_isotonic.joblib` עצמו, מאמתים את סדר הפיצ'רים ואת טביעת ה־SHA-256 שלו, ומעריכים אותו ללא שינוי על אותו Test נעול. אין כאן בחירת מודל או כוונון לפי Test.

In [4]:
model_artifact_path = (
    project_root / "artifacts" / "map_classifier"
    / "canonical_elo_isotonic.joblib"
)
deployed_bundle = joblib.load(model_artifact_path)
assert deployed_bundle["feature_columns"] == feature_columns, (
    "סדר הפיצ'רים בארטיפקט אינו זהה ל־allowlist הנעול."
)
deployed_model = deployed_bundle["calibrated_model"]
model_sha256 = hashlib.sha256(model_artifact_path.read_bytes()).hexdigest()
print(f"ארטיפקט שנפרס: {model_artifact_path.relative_to(project_root)}")
print(f"SHA-256: {model_sha256}")
print(f"סדר פיצ'רים מאומת: {feature_columns}")

ארטיפקט שנפרס: artifacts\map_classifier\canonical_elo_isotonic.joblib
SHA-256: 274a8118ada2c9551f7e668f8999fa0ff672b0158b6d055056a152f09820370d
סדר פיצ'רים מאומת: ['team1_elo_global', 'team2_elo_global', 'team1_elo_map', 'team2_elo_map', 'elo_global_diff', 'elo_map_diff']


## פתיחת הכספת: מדדי Test הסופיים

זהו שלב המדידה החד־פעמי. תחזיות Test משמשות רק לחישוב לוג־לוס, דיוק וברייר בכלל החלוקה ובשכבות מפה 1 ומפות 2+. לא תתבצע בעקבותיהן בחירת מודל נוספת.

בנוסף מדווחים רווחי סמך של 95% באמצעות אלף דגימות Bootstrap ברמת `match_id`. כל מפות המשחק ושני העותקים הסימטריים נדגמים יחד, כדי לשמור על התלות הפנימית ולא להציג אי־ודאות צרה באופן מלאכותי.

In [5]:
test_probability = deployed_model.predict_proba(X_test)[:, 1]
test_prediction = (test_probability >= 0.5).astype(int)

def calculate_test_metrics(mask):
    selected_target = y_test.loc[mask]
    selected_probability = test_probability[mask.to_numpy()]
    selected_prediction = test_prediction[mask.to_numpy()]
    return {
        "rows": int(mask.sum()),
        "accuracy": accuracy_score(selected_target, selected_prediction),
        "brier": brier_score_loss(selected_target, selected_probability),
    }

test_metric_groups = {
    "כלל Test": pd.Series(True, index=test_df.index),
    "מפה 1": test_df["map_position"].eq(1),
    "מפה 2 ומעלה": test_df["map_position"].gt(1),
}
test_metrics = {
    group_name: calculate_test_metrics(mask)
    for group_name, mask in test_metric_groups.items()
}
test_log_loss = log_loss(y_test, test_probability, labels=[0, 1])
print(f"לוג־לוס Test סופי: {test_log_loss:.6f}")
for group_name, metrics in test_metrics.items():
    print(
        f"{group_name}: שורות={metrics['rows']:,}, "
        f"דיוק={metrics['accuracy']:.6f}, ברייר={metrics['brier']:.6f}"
    )

bootstrap_rows = pd.DataFrame({
    "match_id": test_df["match_id"].to_numpy(),
    "rows": 1.0,
    "correct": (test_prediction == y_test.to_numpy()).astype(float),
    "brier_sum": (test_probability - y_test.to_numpy()) ** 2,
    "map1_rows": test_df["map_position"].eq(1).astype(float).to_numpy(),
    "later_rows": test_df["map_position"].gt(1).astype(float).to_numpy(),
})
bootstrap_rows["map1_correct"] = bootstrap_rows["correct"] * bootstrap_rows["map1_rows"]
bootstrap_rows["map1_brier_sum"] = bootstrap_rows["brier_sum"] * bootstrap_rows["map1_rows"]
bootstrap_rows["later_correct"] = bootstrap_rows["correct"] * bootstrap_rows["later_rows"]
bootstrap_rows["later_brier_sum"] = bootstrap_rows["brier_sum"] * bootstrap_rows["later_rows"]
cluster_metrics = bootstrap_rows.groupby("match_id", sort=False).sum()
cluster_values = cluster_metrics.to_numpy(dtype=float)
match_count = len(cluster_values)
bootstrap_rng = np.random.default_rng(RANDOM_SEED)
sample_indices = bootstrap_rng.integers(0, match_count, size=(1000, match_count))
sample_totals = cluster_values[sample_indices].sum(axis=1)

overall_accuracy_samples = sample_totals[:, 1] / sample_totals[:, 0]
overall_brier_samples = sample_totals[:, 2] / sample_totals[:, 0]
map1_accuracy_samples = sample_totals[:, 5] / sample_totals[:, 3]
map1_brier_samples = sample_totals[:, 6] / sample_totals[:, 3]
later_accuracy_samples = sample_totals[:, 7] / sample_totals[:, 4]
later_brier_samples = sample_totals[:, 8] / sample_totals[:, 4]

def confidence_interval(values):
    return np.quantile(values, [0.025, 0.975])

for group_name, accuracy_samples, brier_samples in [
    ("כלל Test", overall_accuracy_samples, overall_brier_samples),
    ("מפה 1", map1_accuracy_samples, map1_brier_samples),
    ("מפה 2 ומעלה", later_accuracy_samples, later_brier_samples),
]:
    accuracy_ci = confidence_interval(accuracy_samples)
    brier_ci = confidence_interval(brier_samples)
    print(
        f"{group_name} — רווח סמך דיוק 95%: "
        f"[{accuracy_ci[0]:.6f}, {accuracy_ci[1]:.6f}]"
    )
    print(
        f"{group_name} — רווח סמך ברייר 95%: "
        f"[{brier_ci[0]:.6f}, {brier_ci[1]:.6f}]"
    )

לוג־לוס Test סופי: 0.646857


כלל Test: שורות=1,190, דיוק=0.666387, ברייר=0.214199
מפה 1: שורות=588, דיוק=0.641156, ברייר=0.222295
מפה 2 ומעלה: שורות=602, דיוק=0.691030, ברייר=0.206291
כלל Test — רווח סמך דיוק 95%: [0.610841, 0.719246]
כלל Test — רווח סמך ברייר 95%: [0.199338, 0.228292]
מפה 1 — רווח סמך דיוק 95%: [0.586735, 0.695578]
מפה 1 — רווח סמך ברייר 95%: [0.208574, 0.235768]
מפה 2 ומעלה — רווח סמך דיוק 95%: [0.626204, 0.753441]
מפה 2 ומעלה — רווח סמך ברייר 95%: [0.189474, 0.223606]


## מסקנה סופית

המדד הרשמי המתוקן של הארטיפקט המכויל שנפרס הוא: לוג־לוס 0.646857, דיוק 0.666387 וברייר 0.214199. במפה 1 התקבלו דיוק 0.641156 וברייר 0.222295; במפות 2+ התקבלו דיוק 0.691030 וברייר 0.206291. רווח הסמך ברמת משחק לדיוק הכולל הוא ‎[0.610841, 0.719246]‎ ולברייר הכולל ‎[0.199338, 0.228292]‎.

מדדי מפה 1 הם האומדן המחמיר והרלוונטי ביותר לסימולציה טרום־סדרתית: רווח הסמך לדיוק הוא ‎[0.586735, 0.695578]‎ ולברייר ‎[0.208574, 0.235768]‎. המספרים הישנים — לוג־לוס 0.615712, דיוק 0.668908 וברייר 0.212750 — מבוטלים, משום שנמדדו על מודל גולמי שאומן מחדש על Train+Validation ולא על הארטיפקט המכויל שמשרת את מערכת החיזוי. המדדים המתוקנים לעיל הם תוצאת ה־holdout הסופית והבלתי ניתנת לכוונון של הפרויקט; לא ייעשה בהם שימוש לשינוי פיצ'רים, פרמטרים או סף החלטה.